# Дообучение ИИ-Тьютора на Qubu GPUКонкурсный демо: Mistral Small 24B + QLoRA 4-bit → Адаптер СПОСпециальность: 15.02.14 «Оснащение средствами автоматизации технологических процессов и производств»Автор: Команда ИИ СИТ (Бардаков Д.Н., Мышанская Н.Г.)

## Шаг 1. Установка зависимостей

In [ ]:
!pip install -q torch transformers accelerate peft bitsandbytes trl datasets sentencepiece protobuf scipyimport torchprint(f"PyTorch: {torch.__version__}")print(f"CUDA available: {torch.cuda.is_available()}")if torch.cuda.is_available():    print(f"GPU: {torch.cuda.get_device_name(0)}")    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## Шаг 2. Загрузка датасетаДатасет должен быть добавлен через меню «Добавить датасет» на Qubu.Если датасет ещё не добавлен, ячейка ниже создаст пробный датасет (5 примеров).

In [ ]:
import json, osDATASET_DIR = "/workspace/dataset"# Проверка датасетаif os.path.exists(f"{DATASET_DIR}/train.jsonl"):    print(f"Датасет найден в {DATASET_DIR}")else:    print(f"Датасет не найден. Создаём пробный...")    os.makedirs(DATASET_DIR, exist_ok=True)        sample_data = [        {"messages": [{"role": "system", "content": "Ты — интеллектуальный тьютор для СПО (15.02.14)."}, {"role": "user", "content": "Сделай конспект лекции о ПЛК."}, {"role": "assistant", "content": "## Конспект: Программируемые Логические Контроллеры (ПЛК)\n\n- **Определение**: ПЛК — это промышленный компьютер для управления технологическими процессами.\n- **Функции**: обработка сигналов с датчиков, логическое управление, взаимодействие.\n- **Языки**: LD (Ladder Diagram), FBD (Function Block Diagram), ST (Structured Text).\n- **Примеры**: Siemens S7, Allen-Bradley ControlLogix, Овен."}], "metadata": {"type": "summary"}},        {"messages": [{"role": "system", "content": "Ты — интеллектуальный тьютор для СПО (15.02.14)."}, {"role": "user", "content": "Что такое PID-регулятор?"}, {"role": "assistant", "content": "PID-регулятор — это устройство автоматического управления, которое поддерживает заданное значение регулируемой величины. Аббревиатура PID расшифровывается как: P — Proportional (пропорциональная), I — Integral (интегральная), D — Derivative (дифференциальная). P-компонент реагирует на текущую ошибку, I-компонент устраняет статическую ошибку, D-компонент обеспечивает предварительное действие. В промышленной автоматизации PID-регуляторы используются для управления температурой, давлением, расходом и другими параметрами."}], "metadata": {"type": "chat"}},        {"messages": [{"role": "system", "content": "Ты — интеллектуальный тьютор для СПО (15.02.14)."}, {"role": "user", "content": "Создай тест по теме SCADA-системы."}, {"role": "assistant", "content": "## Тест: SCADA-системы (3 вопроса, medium)\n\n**1.** Что такое SCADA?\nA) Система управления базами данных  B) Система диспетчерского управления и сбора данных  C) Программа для черчения  D) Операционная система\n**Ответ: B**\n\n**2.** Какая задача НЕ входит в функции SCADA?\nA) Мониторинг  B) Управление  C) Оповещение  D) Воспроизводство музыки\n**Ответ: D**\n\n**3.** Какой протокол НЕ используется в SCADA?\nA) OPC UA  B) Modbus  C) HTTP  D) DNP3\n**Ответ: C"}], "metadata": {"type": "quiz"}},        {"messages": [{"role": "system", "content": "Ты — интеллектуальный тьютор для СПО (15.02.14)."}, {"role": "user", "content": "Сделай конспект по теме датчики давления."}, {"role": "assistant", "content": "## Конспект: Датчики давления\n\n- **Назначение**: измерение давления жидкостей и газов в промышленных процессах.\n- **Типы**: тензометрические, пьезоэлектрические, ёмкостные.\n- **Принцип**: преобразование давления в электрический сигнал.\n- **Область применения**: АСУ ТП, паровые котлы, насосы."}], "metadata": {"type": "summary"}},        {"messages": [{"role": "system", "content": "Ты — интеллектуальный тьютор для СПО (15.02.14)."}, {"role": "user", "content": "Объясни, что такое функциональная безопасность."}, {"role": "assistant", "content": "Функциональная безопасность — это часть общей безопасности системы, зависящая от правильного функционирования системы управления безопасностью (SIS). Основные понятия: SIL (Safety Integrity Level) — уровни полноты безопасности от 1 до 4. Чем выше SIL, тем выше требования к надёжности. Применяется в нефтегазовой, химической и энергетической отраслях для предотвращения аварий."}], "metadata": {"type": "chat"}},    ]        with open(f"{DATASET_DIR}/train.jsonl", "w", encoding="utf-8") as f:        for item in sample_data[:4]:            f.write(json.dumps(item, ensure_ascii=False) + "\n")    with open(f"{DATASET_DIR}/val.jsonl", "w", encoding="utf-8") as f:        f.write(json.dumps(sample_data[4], ensure_ascii=False) + "\n")        print(f"Пробный датасет создан: 4 train + 1 val")# Загрузка датасетовdef load_jsonl(path):    data = []    with open(path, "r", encoding="utf-8") as f:        for line in f:            line = line.strip()            if line:                data.append(json.loads(line))    return datatrain_data = load_jsonl(f"{DATASET_DIR}/train.jsonl")val_data = load_jsonl(f"{DATASET_DIR}/val.jsonl")print(f"Train: {len(train_data)} записей")print(f"Val:   {len(val_data)} записей")

## Шаг 3. Загрузка базовой модели (Mistral Small 24B + QLoRA 4-bit)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfigMODEL_ID = "mistralai/Mistral-Small-24B-Instruct-2501"bnb_config = BitsAndBytesConfig(    load_in_4bit=True,    bnb_4bit_quant_type="nf4",    bnb_4bit_compute_dtype=torch.float16,    bnb_4bit_use_double_quant=True,)print(f"Загрузка модели {MODEL_ID}...")print("Это может занять 5-10 минут.")model = AutoModelForCausalLM.from_pretrained(    MODEL_ID,    quantization_config=bnb_config,    device_map="auto",    trust_remote_code=True,    torch_dtype=torch.float16,    use_cache=False,)model.gradient_checkpointing_enable()tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)if tokenizer.pad_token is None:    tokenizer.pad_token = tokenizer.eos_tokentotal_params = sum(p.numel() for p in model.parameters())print(f"\nМодель загружена: {total_params / 1e9:.1f}B параметров")print(f"VRAM: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

## Шаг 4. Настройка LoRA-адаптера

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_trainingmodel = prepare_model_for_kbit_training(model)lora_config = LoraConfig(    r=16,    lora_alpha=32,    lora_dropout=0.05,    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",                    "gate_proj", "up_proj", "down_proj"],    bias="none",    task_type="CAUSAL_LM",)model = get_peft_model(model, lora_config)trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)print(f"LoRA-адаптер настроен")print(f"Обучаемых параметров: {trainable_params / 1e6:.1f}M")print(f"Доля обучаемых: {trainable_params / total_params * 100:.2f}%")model.print_trainable_parameters()

## Шаг 5. Обучение (SFT)

In [ ]:
import osfrom transformers import TrainingArgumentsfrom trl import SFTTrainerOUTPUT_DIR = "/workspace/lora-adapter"os.makedirs(OUTPUT_DIR, exist_ok=True)TRAINING_MODE = "light"  # debug / light / fulltraining_args = TrainingArguments(    output_dir=OUTPUT_DIR,    num_train_epochs=2 if TRAINING_MODE != "debug" else 1,    per_device_train_batch_size=2,    gradient_accumulation_steps=4,    learning_rate=2e-4,    warmup_ratio=0.1,    lr_scheduler_type="cosine",    logging_steps=10,    save_strategy="steps",    save_steps=100,    eval_strategy="steps",    eval_steps=50,    save_total_limit=2,    load_best_model_at_end=True,    fp16=False,    bf16=torch.cuda.is_bf16_supported(),    gradient_checkpointing=True,    optim="paged_adamw_32bit",    max_grad_norm=1.0,    report_to="none",    remove_unused_columns=False,    max_steps=10 if TRAINING_MODE == "debug" else -1,)print(f"Режим: {TRAINING_MODE}")print(f"Эпохи: {training_args.num_train_epochs}")print(f"Batch: {training_args.per_device_train_batch_size} x {training_args.gradient_accumulation_steps}")

In [ ]:
class TutorCollator:    def __init__(self, tokenizer, max_length=2048):        self.tokenizer = tokenizer        self.max_length = max_length    def __call__(self, examples):        texts = []        for ex in examples:            text = self.tokenizer.apply_chat_template(                ex["messages"], tokenize=False, add_generation_prompt=False            )            texts.append(text)        tokenized = self.tokenizer(            texts, truncation=True, max_length=self.max_length,            padding="longest", return_tensors="pt"        )        tokenized["labels"] = tokenized["input_ids"].clone()        return tokenizedcollator = TutorCollator(tokenizer, max_length=1024 if TRAINING_MODE != "full" else 2048)trainer = SFTTrainer(    model=model,    args=training_args,    train_dataset=train_data,    eval_dataset=val_data if len(val_data) > 0 else None,    tokenizer=tokenizer,    data_collator=collator,)print("Запуск обучения...")trainer.train()print("Обучение завершено!")

## Шаг 6. Сохранение адаптера

In [ ]:
adapter_path = os.path.join(OUTPUT_DIR, "final")trainer.save_model(adapter_path)tokenizer.save_pretrained(adapter_path)print(f"Адаптер сохранён: {adapter_path}")import subprocessresult = subprocess.run(["ls", "-lh", adapter_path], capture_output=True, text=True)print(f"\nФайлы:{result.stdout}")

## Шаг 7. Тестирование адаптера

In [ ]:
from peft import PeftModelprint("Загрузка модели с адаптером...")base_model = AutoModelForCausalLM.from_pretrained(    MODEL_ID, quantization_config=bnb_config,    device_map="auto", torch_dtype=torch.float16)model_with_adapter = PeftModel.from_pretrained(base_model, adapter_path)model_with_adapter.eval()# Тест 1: Конспектtest_prompt = tokenizer.apply_chat_template(    [{"role": "user", "content": "Сделай краткий конспект лекции о пневматике и гидравлике."}],    tokenize=False, add_generation_prompt=True)inputs = tokenizer(test_prompt, return_tensors="pt").to(model_with_adapter.device)with torch.no_grad():    outputs = model_with_adapter.generate(**inputs, max_new_tokens=200, temperature=0.7, top_p=0.9)response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)print(f"\nТест 1 - Конспект:{response[:500]}")# Тест 2: Тестtest_prompt2 = tokenizer.apply_chat_template(    [{"role": "user", "content": "Создай 3 вопроса по теме температурных датчиков."}],    tokenize=False, add_generation_prompt=True)inputs2 = tokenizer(test_prompt2, return_tensors="pt").to(model_with_adapter.device)with torch.no_grad():    outputs2 = model_with_adapter.generate(**inputs2, max_new_tokens=300, temperature=0.8, top_p=0.9)response2 = tokenizer.decode(outputs2[0][inputs2["input_ids"].shape[1]:], skip_special_tokens=True)print(f"\nТест 2 - Тест:{response2[:500]}")

## ИтогиАдаптер сохранён в .Для настройки инференса на Qubu:1. Загрузите адаптер как артефакт ноутбука2. В настройках инференса укажите 3. service.py автоматически подгрузит адаптер